In [ ]:
# import numpy and matplotlib 
import numpy as np
import matplotlib.pyplot as plt

# **Metodo agli Elementi Finiti (stazionario) - Parte 1**

Il metodo agli elementi finiti (FEM) è una tecnica di risoluzione numerica per equazioni alle derivate parziali, basata sulla discretizzazione di domini spaziali attraverso mesh poligonali (spesso e volentieri triangolari). Nel caso mono-dimensionale, in particolare, ciò si riduce all'introduzione di griglie spaziali.

La peculiarità del FEM è quella di risolvere il problema differenziali in *forma debole*, cioè passando da un'equazione puntuale (definita per ogni $x$ nel dominio), ad una variazionale (definita per ogni funzione test $v$). Per fare ciò, si fa leva su alcuni concetti di Analisi Funzionale, quali: spazi funzionali (Sobolev e Lebesgue), norme integrali, prodotti interni, forme bilineari, funzionali lineari, etc.

### Discretizzazione agli elementi finiti - **Mesh**

L'idea alla base del FEM è quella di discretizzare il dominio spaziale $\Omega$ introducendo una mesh $\mathcal{M}$. Scelto un grado polinomiale $r$, quest'ultima viene utilizzata per costruire uno spazio elementi finiti

$$V_{h}\subset V,$$

caratterizzato da tutte quelle funzioni
$v_h:\Omega\to\mathbb{R}$ che sono polinomiali a tratti (di grado $r$), cioè, limitatamente ad ogni elemento della mesh, si possono scrivere come polinomi di grado $r$.

Nel caso Lagrangiano, questa costruzione è automaticamente associata, ad una collezione di nodi, $x_{1},\dots,x_{N_h}$, detti *gradi di libertà* (dofs). Questi ultimi, infatti, servono per l'interpolazione locale, che avviene elemento per elemento (similmente alle spline).

In [ ]:
# importo la classe Grid 
from fem_utils import Grid

In [ ]:

## Costruzione della griglia e rappresentazione grafica nell'intervallo [0, 1]
# Definisco gli estremi del dominio
a, b = 0, 1
# definisco la dimensione degli elementi
h = 0.25
# costruisco la griglia

# rappresento la griglia graficamente


### Discretizzazione agli elementi finiti - **Funzioni**

Il vantaggio principale è che ogni funzione $f_h \in V_h$ si può rappresentare **univocamente** attraverso il vettore dei suoi valori nodali $\mathbf{f}_h$. In altre parole, esiste una corrispondenza biunivoca (uno-a-uno) tra $f_h$ e $\mathbf{f}_h$.


$$V_h\ni f_h\;\;\longleftrightarrow\;\;\mathbf{f}_{h}\in \mathbb{R}^{N_h}$$

dove $\mathbf{f}_{h}=[f_h(x_1),\dots,f_h(x_{N_h})]$ è il vettore di valutazione della funzione $f_h$ nei valori nodali (nei gradi di libertà).

In [ ]:
from fem_utils import fun2dof

# funzione continua
f = lambda x: np.sin(2 * np.pi * x) * x
# punti per rappresentazione grafica
xplot = np.linspace(a, b, 1000)

# valutazione della funzione nei gradi di liberta (nodi della griglia)

# genero una funzione nello spazio FE utilizzando i valori nodali (gradi di liberta) di f

plt.figure(figsize=(4, 3))
# rappresentazione grafica funzione continua
plt.plot(xplot, f(xplot), "--m", label="$f$", alpha=0.4)

plt.legend()
plt.show()

In [ ]:
# stampiamo la posizione dei nodi

# numero dei nodi

# funzione discreta a funzione continua evaluata in uno dei dof
# print(fh(0.25), f(0.25))
# funzione discreta a funzione continua evaluata in un punto generico
# print(fh(0.2), f(0.2))


<mark>**Esercizio 1:** funzioni di base</mark></br>

Le funzioni di base $\varphi_{j}\in V_{h}$ sono quelle funzioni la cui rappresentazione in vettore dof corrisponde ai vettori della base canonica $\mathbf{e}_{j}=[0,0,\dots,1,\dots,0,0]$, dove "l'1" è in posizione $j$.

Si consideri la terza funzione di base, $j=3$
1. Rappresentare graficamente $\varphi_j$,
2. Determinare $x$ tale che $\varphi_{j}(x)=1$.

NB: non fate confusione con l'indicizzazione, ricordate che in Python partiamo a contare da zero!

In [ ]:
# costruzione dei dofs

# rappresentazione grafica
plt.figure(figsize=(4, 3))


plt.title("Terza funzione test")
plt.show()

## Elementi finiti per problemi ellittici

Il FEM ci permette di risolvere equazioni differenziali (lineari) trasformandole in problemi algebrici (sistemi lineari). Vediamolo con un esempio.
</br>
</br>
Sia $\Omega=(a,b)$. Vogliamo risolvere il problema

$$-u'' = f \quad \text{in}\;\Omega,$$

complementato da condizioni di Dirichlet (dbc), $u(a)=\alpha$, $u(b)=\beta$, ai bordi del dominio. Abbiamo

- **Formulazione forte**: trovare $u\in \mathcal{C}^{2}(\Omega)$ soddisfacente le dbc e tale che</br></br>
$$ -u''(x)=f(x)\quad\forall x\in\Omega. $$

- **Formulazione debole**: trovare $u\in H^{1}(\Omega)$ soddisfacente le dbc e tale che</br></br>
$$\int_a^bu'v'dx=\int_a^bfvdx\quad\forall v\in H_0^1(\Omega).$$

- **Problema di Galerkin**: trovare $u_h\in V_h$ soddisfacente le dbc e tale che</br></br>
$$\int_a^bu_h'v_h'dx=\int_a^bfv_hdx\quad\forall v_h\in V_h\cap H_0^1(\Omega).$$

A questo punto possiamo riscrivere il problema di Galerkin in formulazione algebrica. Innanzitutto osserviamo che ogni funzione di $V_h$ puo essere scritta come combinazione lineare delle funzioni di base. Pertanto
$$ 
u_h = \sum_{i=1}^N u_{h,i}\varphi_i 
$$ 
Concentriamoci sul nodo j-esimo e scegliamo $v_h =\varphi_j$ (tanto il problema di Galerkin e soddisfatto $\forall v_h \in V_h \cap H_0^1(\Omega)$). Sostituendo nel problema di Galerkin otteniamo:
$$ 
\sum_{i=1}^N u_{h,i}\int_{\Omega} \varphi_i' \varphi_j'dx = \int_{\Omega} f \varphi_j 
$$</br>
Scrivendo un'equazione di questo tipo per ogni $j$ e raccogliendo tutte le equazioni all'interno di un sistema otteniamo la...

- **Formulazione algebrica**: trovare $\mathbf{u}_h\in\mathbb{R}^{N_h}$ soddisfacente le dbc e tale che</br></br>
$$\mathbf{A}\mathbf{u}_h = \mathbf{F},$$
dove 
$$
\mathbf{u}_h = [u_{h,1},~u_{h,2},...,u_{h,N}]^T, \quad A_{ji} = \int_\Omega \varphi_i' \varphi _j' dx, \quad F_j = \int_\Omega f\varphi_j dx
$$

- **Formulazione algebrica (con dbc)**: trovare $\mathbf{u}_h\in\mathbb{R}^{N_h}$ tale che</br></br>
$$\tilde{\mathbf{A}}\mathbf{u}_h = \tilde{\mathbf{F}}.$$

L'ultimo step si ottiene modificando $\mathbf{A}$ e $\mathbf{F}$ in maniera opportuna, cosi da includere le condizioni al bordo direttamente nel sistema.

<mark>**Esercizio 3**</mark></br>

Si consideri il seguente problema ellittico,</br></br>

$$\begin{cases}-u'' = e^{2x}\left(3\sin x + 4\cos x\right) & x\in(0,2\pi)\\
\\u(0)=u(2\pi)=0,
\end{cases}$$
Si risolva numericamente il problema differenziale implementando il metodo agli elementi finiti con $h=0.01$.

In [ ]:
# Esercizio 3
# dati del problema e costruzione della griglia
a = 0.0
b = 2 * np.pi
h = 0.01
# fuzione di carico
f = lambda x: np.exp(2 * x) * (3 * np.sin(x) + 4 * np.cos(x))

# generazione griglia


### 1) Struttura della matrice di diffusione

Nel caso 1D con elementi lineari e mesh uniforme la matrice di rigidezza è **tridiagonale**:
$$
\mathbf{A} = \frac{1}{h}
\begin{bmatrix}
1 & -1 & 0 & \cdots & 0\\
-1 & 2 & -1 & \ddots & \vdots\\
0 & -1 & 2 & \ddots & 0\\
\vdots & \ddots & \ddots & \ddots & -1\\
0 & \cdots & 0 & -1 & 1
\end{bmatrix}.
$$

### 2) Assemblaggio del termine noto con la matrice di massa

Il termine noto è un vettore
$$F = [F_0, F_1, \dots, F_N]^T,$$
dove
$$F_j = \int_a^b f(x)\,\varphi_j(x)\,dx.$$

Approssimando $f$ con $f_h \in V_h$, otteniamo
$$F_j \approx \int_a^b f_h(x)\,\varphi_j(x)\,dx.$$

Scrivendo $f_h = \sum_{i=0}^N (f_h)_i\varphi_i$, segue
$$F_j \approx \sum_{i=0}^N (f_h)_i\int_a^b \varphi_i(x)\varphi_j(x)\,dx,$$
cioe in forma matriciale
$$\mathbf{F} \approx \mathbf{M}\,\mathbf{f}_h,$$
dove $\mathbf{M}$ è detta matrice di massa e $\mathbf{f}_h=[f(x_0),\dots,f(x_N)]^T$ è il vettore dei valori nodali.

Per elementi P1 su mesh uniforme, la forma esplicita è
$$
\mathbf{M} = h
\begin{bmatrix}
\frac{1}{3} & \frac{1}{6} & 0 & \cdots & 0\\
\frac{1}{6} & \frac{2}{3} & \frac{1}{6} & \ddots & \vdots\\
0 & \frac{1}{6} & \frac{2}{3} & \ddots & 0\\
\vdots & \ddots & \ddots & \ddots & \frac{1}{6}\\
0 & \cdots & 0 & \frac{1}{6} & \frac{1}{3}
\end{bmatrix}
= \frac{h}{6}
\begin{bmatrix}
2 & 1 & 0 & \cdots & 0\\
1 & 4 & 1 & \ddots & \vdots\\
0 & 1 & 4 & \ddots & 0\\
\vdots & \ddots & \ddots & \ddots & 1\\
0 & \cdots & 0 & 1 & 2
\end{bmatrix}.
$$

In [ ]:
# assemblaggio della matrice di rigidezza A
from fem_utils import diffusion


In [ ]:
# assemblaggio del termine noto F con la matrice di massa
from fem_utils import fun2dof, mass


### 3) Imporre le condizioni di Dirichlet

L'operatore `R` ci permette di selezionare i gradi di libertà, passando dal sistema completo al sistema ridotto
$$A_0 = R A R^T, \qquad F_0 = R\,(F - A\,u_{bc}).$$

Una volta risolto $A_0 u_0 = F_0$, la soluzione completa si ricostruisce con
$$u = u_{bc} + R^T u_0.$$

In pratica:
- `R` elimina le righe/colonne associate ai nodi di Dirichlet,
- `R.T` reinserisce i gradi di liberta interni nella soluzione globale.

In [ ]:
from fem_utils import create_restriction

# Definizione degli indici e dei valori per le condizioni al bordo di tipo Dirichlet

# Costruzione del vettore dei valori al bordo

# Assegnazione dei valori di Dirichlet ai nodi corrispondenti


# Costruzione matrice R
# costruzione di un vettore di booleani 
keep_dof = np.ones(Nele + 1, dtype=bool)
# impostazione a False dei dof corrispondenti ai nodi di Dirichlet
keep_dof[dirichlet_nodes] = False
# creazione matrice di restrizione R
R = create_restriction(keep_dof)


# Restrizione di A e del termine noto

# risoluzione del sistema lineare per i dof interni

# ricostruzione della soluzione completa aggiungendo i valori al bordo


# rappresentazione grafica della soluzione numerica
plt.figure(figsize=(4, 3))
plt.plot(grid.nodes, u, marker=".")
plt.title("soluzione numerica")
plt.show()

<mark>**Esercizio 4**</mark></br>

Si consideri il problema alle derivate parziali descritto precedentemente. La soluzione esatta di tale problema è

$$u(x)=-e^{2x}\sin(x).$$

Avendo fissato il grado polinomiale della discretizzazione agli elementi finiti, $r=1$, si calcoli l'errore in norma $L^{2}$ tra la soluzione FEM e la soluzione esatta al variare del passo di discretizzazione $h=0.2,0.1,0.05,0.025$. Plottare graficamente l'andamento dell'errore: i risultati sono coerenti con la teoria?

In [ ]:
from fem_utils import error_l2

# soluzione esatta
uex = lambda x: -np.exp(2 * x) * np.sin(x)
# soluzione approssimata

plt.figure(figsize=(4, 3))
plt.plot(grid.nodes, uex(grid.nodes), marker=".", label="soluzione esatta")
#plt.plot(# todo, #todo, marker=".", label="soluzione numerica")
plt.legend()
plt.show()

err = ## 
print("err =", err)

In [ ]:
from fem_utils import diffusion, fun2dof, mass

# Grado di polinomio
r = 1
# h target
h_target = np.array([0.2, 0.1, 0.05, 0.025])
# h ed errori
h = []
errors = []
# estremi intervallo
a = 0
b = 2 * np.pi
# ciclo for

    # Generazione griglia

    # salvataggio del valore di h della griglia

    # assemblaggio della matrice di rigidezza A

    # assemblaggio del termine noto F con la matrice di massa

    # Definizione degli indici e dei valori per le condizioni al bordo di tipo Dirichlet

    # Costruzione del vettore dei valori al bordo

    # Costruzione matrice R
    keep_dof = np.ones(Nele + 1, dtype=bool)
    keep_dof[dirichlet_nodes] = False
    R = create_restriction(keep_dof)
    
    # Restrizione di A e del termine noto
  
    # risoluzione del sistema lineare per i dof interni
  
    # ricostruzione della soluzione completa aggiungendo i valori al bordo
  
    # calcolo dell'errore L2


    # salvataggio dell'errore
    errors.append(err)

# trasformo liste in vettori
errors = np.array(errors)
h = np.array(h)

print(errors)

In [ ]:
import matplotlib.pyplot as plt

# rappresentazione grafica dell'errore in scala log-log
C = errors.max()
plt.loglog(h, errors, label="$L^2$ error")
plt.loglog(h, C * h ** (r + 1), "--", label="$Ch^%d$" % (r + 1))
plt.loglog(h, C * h, "--", label="$Ch$")
plt.legend()
plt.title("Decadimento dell'errore ($r$ = %d)" % r)
plt.xlabel("Stepsize $h$")
plt.ylabel("Errore")
plt.show()